In [1]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, dash_table
from dash.dependencies import Input, Output

# ============================================
# CARGA DE DATOS
# ============================================
ruta = r"C:\Users\Maestría IA\Documents\Aplicaciones\Actividad4"

df_mortalidad = pd.read_excel(f"{ruta}\Anexo1.NoFetal2019_CE_15-03-23.xlsx")
df_codigos = pd.read_excel(f"{ruta}\Anexo2.CodigosDeMuerte_CE_15-03-23.xlsx")
df_divipola = pd.read_excel(f"{ruta}\Divipola_CE_.xlsx")

# Limpieza básica
df_mortalidad.columns = df_mortalidad.columns.str.strip().str.upper()

# ============================================
# MAPA DE MUERTES POR DEPARTAMENTO
# ============================================
muertes_departamento = df_mortalidad.groupby("DPTO_OCUR").size().reset_index(name="TOTAL_MUERTES")
fig_mapa = px.choropleth(
    muertes_departamento,
    geojson=None,
    locations="DPTO_OCUR",
    color="TOTAL_MUERTES",
    title="Distribución total de muertes por departamento (2019)",
    color_continuous_scale="Reds"
)

# ============================================
# GRÁFICO DE LÍNEAS (MUERTES POR MES)
# ============================================
muertes_mes = df_mortalidad.groupby("MES_OCURR").size().reset_index(name="TOTAL_MUERTES")
fig_lineas = px.line(
    muertes_mes,
    x="MES_OCURR",
    y="TOTAL_MUERTES",
    markers=True,
    title="Total de muertes por mes en Colombia (2019)"
)

# ============================================
# 5 CIUDADES MÁS VIOLENTAS (Códigos X95)
# ============================================
df_homicidios = df_mortalidad[df_mortalidad["CAUSA_DEF"].astype(str).str.startswith("X95")]
top_violentas = df_homicidios["MUN_OCUR"].value_counts().head(5).reset_index()
top_violentas.columns = ["CIUDAD", "TOTAL_HOMICIDIOS"]
fig_barras_violentas = px.bar(
    top_violentas,
    x="CIUDAD",
    y="TOTAL_HOMICIDIOS",
    title="5 ciudades más violentas (Códigos X95)"
)

# ============================================
# 10 CIUDADES CON MENOR ÍNDICE DE MORTALIDAD
# ============================================
muertes_ciudad = df_mortalidad.groupby("MUN_OCUR").size().reset_index(name="TOTAL_MUERTES")
ciudades_menor = muertes_ciudad.nsmallest(10, "TOTAL_MUERTES")
fig_circular = px.pie(
    ciudades_menor,
    values="TOTAL_MUERTES",
    names="MUN_OCUR",
    title="10 ciudades con menor índice de mortalidad (2019)"
)

# ============================================
# TABLA DE LAS 10 PRINCIPALES CAUSAS DE MUERTE
# ============================================
causas = df_mortalidad.groupby("CAUSA_DEF").size().reset_index(name="TOTAL_CASOS")
causas = causas.merge(df_codigos, left_on="CAUSA_DEF", right_on="CODIGO", how="left")
causas_top10 = causas.sort_values("TOTAL_CASOS", ascending=False).head(10)[["CAUSA_DEF", "NOMBRE", "TOTAL_CASOS"]]

# ============================================
# GRÁFICO DE BARRAS APILADAS (MUERTES POR SEXO Y DEPARTAMENTO)
# ============================================
muertes_sexo_dep = df_mortalidad.groupby(["DPTO_OCUR", "SEXO"]).size().reset_index(name="TOTAL_MUERTES")
fig_barras_apiladas = px.bar(
    muertes_sexo_dep,
    x="DPTO_OCUR",
    y="TOTAL_MUERTES",
    color="SEXO",
    title="Muertes por sexo y departamento (2019)",
    barmode="stack"
)

# ============================================
# HISTOGRAMA (GRUPO DE EDAD)
# ============================================
df_mortalidad["GRUPO_EDAD1"] = pd.to_numeric(df_mortalidad["GRUPO_EDAD1"], errors="coerce")

rangos = {
    "Mortalidad neonatal": range(0, 5),
    "Mortalidad infantil": range(5, 7),
    "Primera infancia": range(7, 9),
    "Niñez": range(9, 11),
    "Adolescencia": [11],
    "Juventud": range(12, 14),
    "Adultez temprana": range(14, 17),
    "Adultez intermedia": range(17, 20),
    "Vejez": range(20, 25),
    "Longevidad / Centenarios": range(25, 29),
    "Edad desconocida": [29]
}

def asignar_rango(codigo):
    for k, v in rangos.items():
        if codigo in v:
            return k
    return "Desconocido"

df_mortalidad["RANGO_EDAD"] = df_mortalidad["GRUPO_EDAD1"].apply(asignar_rango)

fig_histograma = px.histogram(
    df_mortalidad,
    x="RANGO_EDAD",
    title="Distribución de muertes por rango de edad (2019)"
)

# ============================================
# APLICACIÓN DASH
# ============================================
app = Dash(__name__)
app.title = "Mortalidad en Colombia 2019"

app.layout = html.Div([
    html.H1("Análisis de Mortalidad en Colombia - 2019", style={"textAlign": "center"}),

    dcc.Graph(figure=fig_mapa),
    dcc.Graph(figure=fig_lineas),
    dcc.Graph(figure=fig_barras_violentas),
    dcc.Graph(figure=fig_circular),

    html.H3("10 principales causas de muerte en Colombia (2019)"),
    dash_table.DataTable(
        data=causas_top10.to_dict("records"),
        columns=[{"name": i, "id": i} for i in causas_top10.columns],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px'}
    ),

    dcc.Graph(figure=fig_barras_apiladas),
    dcc.Graph(figure=fig_histograma)
])

if __name__ == '__main__':
    app.run_server(debug=True)


ModuleNotFoundError: No module named 'plotly'